# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deepikachebrolu/flyrank-ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

### Provisional lane: CTR / Engagement Opportunity Scoring

I am choosing the CTR / Engagement Opportunity Scoring lane because it connects observable search and engagement signals to a concrete decision: which visible pages may deserve review because they appear to under-capture clicks or engagement. The starter dataset contains signals such as impressions, clicks, CTR, average position, sessions, engagement, content age, freshness, intent, and content type. These signals provide a basis for comparing pages with meaningful visibility and identifying potential opportunities for further review.

This lane is provisional because low CTR or engagement can have multiple explanations, including position, low volume, search intent, seasonality, or other factors. The next stages of analysis will test whether a position-adjusted comparison and minimum-volume filters produce a useful ranking.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

### Research question

Which visible pages under-capture clicks or engagement and therefore deserve metadata, content, or monitoring review?

### Unit of analysis

The unit of analysis is one pseudonymized content item/page.

### Decision

The decision is which visible pages should be prioritized for CTR or engagement review, while accounting for search position and minimum traffic volume.

### Action

A reviewer could inspect a high-priority page and decide whether to review its title or metadata, improve intent alignment or snippet structure, improve on-page engagement, or simply monitor it. The ranking is intended to support human review rather than automatically determine what change should be made.

### Cost of a wrong recommendation

A false positive could cause a reviewer to spend time investigating a page that does not have a meaningful CTR or engagement opportunity. A false negative could cause a page with a genuine opportunity to be overlooked. Low-volume pages also create a risk of treating random variation as a meaningful signal, so minimum-volume filters will be important.

### Why data or ML could help

CTR and engagement depend on several observable factors, including impressions, clicks, search position, sessions, content age, intent, and content type. Comparing pages while accounting for these factors may reveal opportunities that are difficult to identify consistently through simple inspection. A position-adjusted baseline can provide a transparent starting point, after which a ranking model can be evaluated if it adds useful signal.

### Provisional evaluation metric

If this project develops into a ranked review queue, I will use Precision@K as a primary evaluation metric because the practical use case is prioritizing a limited number of pages for human review. The value of K will depend on the assumed review capacity.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Unique content items:", df["content_id"].nunique())

print("\nRelevant columns:")
print([
    col for col in [
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "sessions_90d",
        "engagement_rate",
        "scroll_rate",
        "content_age_days",
        "content_type",
        "main_intent"
    ]
    if col in df.columns
])

Rows: 30000
Columns: 44
Unique content items: 30000

Relevant columns:
['impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'sessions_90d', 'engagement_rate', 'scroll_rate', 'content_age_days', 'content_type', 'main_intent']


In [12]:
import pandas as pd

# Only include pages with meaningful search visibility
visible = df[
    (df["impressions_90d"] >= 500) &
    (df["avg_position"] > 0) &
    (df["avg_position"] <= 20)
].copy()

print("Total content items:", len(df))
print("Visible pages (500+ impressions, positions 1-20):", len(visible))

# Check the CTR measurement
print("\nCTR statistics:")
print(visible["ctr"].describe())

# Check engagement measurements
print("\nEngagement statistics:")
print(df["engagement_rate"].describe())

# Inspect missing values
print("\nMissing values:")
print(
    df[
        ["impressions_90d", "ctr",
         "avg_position", "engagement_rate"]
    ].isna().sum()
)

Total content items: 30000
Visible pages (500+ impressions, positions 1-20): 12023

CTR statistics:
count    12023.000000
mean         0.312079
std          0.343159
min          0.000000
25%          0.100000
50%          0.210000
75%          0.410000
max          5.430000
Name: ctr, dtype: float64

Engagement statistics:
count    30000.000000
mean         2.534520
std          8.310096
min          0.000000
25%          0.000000
50%          0.000000
75%          1.350000
max        100.000000
Name: engagement_rate, dtype: float64

Missing values:
impressions_90d    0
ctr                0
avg_position       0
engagement_rate    0
dtype: int64


In [13]:
print("CTR percentiles:")
print(
    visible["ctr"].quantile(
        [0, 0.25, 0.50, 0.75, 0.90, 1]
    )
)

print("\nEngagement percentiles:")
print(
    df["engagement_rate"].quantile(
        [0, 0.25, 0.50, 0.75, 0.90, 1]
    )
)

CTR percentiles:
0.00    0.00
0.25    0.10
0.50    0.21
0.75    0.41
0.90    0.71
1.00    5.43
Name: ctr, dtype: float64

Engagement percentiles:
0.00      0.00
0.25      0.00
0.50      0.00
0.75      1.35
0.90      6.94
1.00    100.00
Name: engagement_rate, dtype: float64


In [14]:
low_ctr_visible = visible[visible["ctr"] < 0.5]

print("Visible pages:", len(visible))
print("Visible pages with CTR < 0.5:", len(low_ctr_visible))
print(
    "Percentage of visible pages with CTR < 0.5:",
    round(len(low_ctr_visible) / len(visible) * 100, 2),
)

Visible pages: 12023
Visible pages with CTR < 0.5: 9759
Percentage of visible pages with CTR < 0.5: 81.17


### What these numbers suggest

The starter dataset contains 30,000 unique content items across 44 columns. Using an initial visibility filter of at least 500 impressions over the 90-day window and an average position between 1 and 20, 12,023 pages qualify for further CTR opportunity analysis.

Among these visible pages, 9,759 have a CTR below 0.5%, representing 81.17% of the filtered pages. This suggests that there is a sufficiently large group of visible pages to investigate whether some pages under-capture clicks.

The result is only an initial screening signal. Low CTR does not by itself show that a page has a poor title, metadata, content, or search-intent match. CTR is also affected by position, query mix, volume, and other factors, so the next stages should use position-aware comparisons and appropriate volume filters.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

### What I can claim

- The starter dataset contains observable search and engagement signals.
- 12,023 pages meet the initial visibility and position filter used for this exploration.
- 9,759 of those filtered pages have CTR below 0.5%.
- This provides a sufficiently large group for further investigation of CTR opportunity.
- I can investigate whether some pages under-capture clicks or engagement relative to appropriate comparison groups.
- The eventual output can be used to prioritize pages for human review.

### What I cannot claim

- I cannot claim that low CTR proves a title or meta description is the cause.
- I cannot claim that changing metadata will definitely increase clicks.
- I cannot claim that a page with low CTR is necessarily underperforming without accounting for position, volume, intent, and other context.
- I cannot claim that this analysis proves a Google ranking factor.
- I cannot claim that the ranking will be useful until it is properly validated.
- I cannot claim that the initial CTR threshold of 0.5% is a universal definition of poor performance; it is a provisional screening threshold that should be tested.
- I cannot claim that a refresh or other intervention caused an improvement without an appropriate causal design.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Self-check

- [x] I selected a predefined lane: CTR / Engagement Opportunity Scoring.
- [x] I explained why I selected this lane.
- [x] I defined the research question.
- [x] I defined the unit of analysis as one content item/page.
- [x] I defined the decision.
- [x] I defined the possible action.
- [x] I explained the cost of a wrong recommendation.
- [x] I explained why data or ML could help.
- [x] I used real numbers from the starter dataset.
- [x] I considered position and minimum-volume issues.
- [x] I stated what I can and cannot claim.
- [x] I avoided unsupported causal claims.
- [x] I executed the complete notebook successfully.
- [ ] I committed the notebook to Git.
- [ ] I pushed the notebook to GitHub.
